#### 감정 분석 자연어 처리
1. data 폴더 안에 ratings_train.txt 파일을 로드
2. 데이터를 상위 500개 데이터만 추출
3. 리뷰 데이터와 감정 데이터로 나눠준다.
4. 리뷰 데이터를 토큰화(komoran함수 이용) -> 벡터화(Word2Vec, 단위 벡터의 평균)
5. Word2Vec 학습
    - window -> 3
    - epochs -> 10
    - min_count -> 5
    - sg -> 1
    - seed -> 42
6. 벡터화(Word2Vec, 단위 벡터의 평균)
7. 분류 모델(SVC, Logistic)
8. train, test을 이용하여 2개의 모델 중 성능이 높은 모델이 무엇인가?
9. 단위 벡터의 평균의 성능과 단위 벡터 + 중요도 평균의 성능의 차이를 확인

In [1]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv('../data/ratings_train.txt', sep='\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [4]:
# 상위 데이터 500개
# df2 = df.head(500)
df2 = df.loc[:499, ]
df2

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
495,10049872,그냥 기독교영화네요. 좀 더 깊이있는 내용을 기대했는데.. 실망입니다. 영화도 뭔가...,0
496,9427493,그냥 책으로 읽는게 더 절절하게 다가온다. 영화는 내용을 너무 비약하고 삭제해서 행...,0
497,6629097,너무나 감동적인 영화,1
498,9995177,스킨헤드성님들이 이 영화를 싫어합니다.,0


In [5]:
df2['label'].value_counts()

label
1    258
0    242
Name: count, dtype: int64

In [6]:
def bulid_tokenize():
    try:
        # 라이브러리 로드 -> 라이브러리가 존재하면 코드를 실행
        from konlpy.tag import Komoran
        komoran = Komoran()
        allow_pos = ['NNP' ,'NNG', 'VV', 'VA', 'SL', 'MAG']
        def tokenize(text):
            tokens = []
            for word, pos in komoran.pos(text):
                if pos in allow_pos:
                    tokens.append(word)
            return tokens
        # tokenize 함수를 결과로 되돌려준다.
        return tokenize
    except Exception as e:
        print("Komoran 사용 불가 :", e)
        return lambda x : x.split()

In [7]:
tokenize = bulid_tokenize()

In [8]:
reviews = df2['document'].values
Y = df2['label'].values

In [9]:
X_tokens = [tokenize(review) for review in reviews]

In [10]:
w2v = Word2Vec(
    sentences=X_tokens,
    window=5,
    min_count=2,
    sg = 1,
    epochs=100,
    seed=42
)

In [11]:
wv = w2v.wv

In [12]:
def sent_embed_mean(tokens):
    vecs = []
    for word in tokens:
        if word in wv.index_to_key:
            vecs.append(wv[word])
    result =  np.mean(vecs, axis=0) if vecs else np.zeros(wv.vector_size)
    return result

In [13]:
tfidf_vec = TfidfVectorizer(
    tokenizer=tokenize,
    lowercase=False
).fit(reviews)

idf = dict(
    zip(
        # get_feature_names_out() -> Tfidf에서 사용된 단어들의 목록
        tfidf_vec.get_feature_names_out(),
        # idf_ -> 중요도
        tfidf_vec.idf_
    )
)

# 단어 별 단위 벡터의 평균과 idf을 곱한다. 
def sent_embed_tfidf(tokens):
    vecs = []
    weight = []
    for word in tokens:
        # tokens에 각각의 단어가 Word2Vec과 TF-IDF에 존재한다면
        if word in wv.key_to_index and word in idf:
            # vecs -> 단위벡터와 중요도를 곱한 값을 vecs 추가
            vecs.append(wv[word] * idf[word])
            # weight -> 중요도 데이터를 추가 
            weight.append(idf[word])
    # vecs의 데이터가 존재하지 않는다면 -> tokens 안에 단어는 존재하지만 Word2Vec이나
    # TD-IDF에 단어가 존재하지 않을때
    if not vecs:
        # 희소 행렬 되돌려준다. 0행렬
        result = np.zeros(wv.vector_size)
    else:
        result = np.sum(vecs, axis=0) / ( np.sum(weight) + 1e-9 )
    return result

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [14]:
X_embed = [sent_embed_mean(token) for token in X_tokens]
X_embed

[array([ 5.33957541e-01,  7.00074423e-04, -3.38822186e-01,  5.87574253e-03,
         2.37580147e-02,  5.98014832e-01,  1.17120527e-01,  1.48372784e-01,
        -3.04113656e-01, -8.77798945e-02, -5.33828497e-01,  1.42356157e-01,
         1.34550840e-01,  3.23668033e-01, -2.32036397e-01,  2.44896889e-01,
         4.58733812e-02, -2.46577501e-01,  7.21570104e-02, -6.67451173e-02,
        -6.07907474e-02,  1.77631378e-01,  9.34182256e-02,  2.20190808e-01,
         4.89958584e-01,  8.94318894e-02,  6.21634051e-02,  9.69333500e-02,
         2.22539753e-01, -1.55158252e-01, -2.60440171e-01,  4.50167321e-02,
         2.24932097e-02,  3.69715592e-04, -1.07274912e-01, -3.40299234e-02,
         2.51285970e-01, -2.95914114e-01,  1.00796878e-01, -7.39332512e-02,
         2.45106518e-02, -3.59761491e-02,  4.87316065e-02,  1.47817567e-01,
         1.73935026e-01,  3.09559405e-02,  1.07413985e-01,  5.93005009e-02,
         3.50145280e-01, -2.59373002e-02, -3.90245058e-02, -7.06655800e-01,
         1.4

In [15]:
X_embed2 = [sent_embed_tfidf(token) for token in X_tokens]
X_embed2

[array([ 5.39810571e-01, -1.63267108e-02, -3.34723341e-01,  2.10780082e-04,
         1.28492306e-02,  5.96225796e-01,  9.78287297e-02,  1.56600010e-01,
        -2.93080290e-01, -8.62237030e-02, -5.29292403e-01,  1.24439640e-01,
         1.32396636e-01,  3.34281594e-01, -2.39801207e-01,  2.42105018e-01,
         4.63566677e-02, -2.38939629e-01,  6.36984335e-02, -6.13891317e-02,
        -6.15614971e-02,  1.80698753e-01,  8.58029122e-02,  2.22408567e-01,
         4.86226051e-01,  8.98643890e-02,  6.28417737e-02,  1.20808420e-01,
         2.24022601e-01, -1.45166678e-01, -2.53360332e-01,  3.45796081e-02,
         2.60688084e-02,  2.58241135e-05, -1.07769559e-01, -2.92493854e-02,
         2.48378824e-01, -2.95590640e-01,  8.87420428e-02, -6.94806043e-02,
         2.68022020e-02, -4.16400678e-02,  4.63562419e-02,  1.48776384e-01,
         1.74009445e-01,  2.84692285e-02,  1.09732941e-01,  7.13947251e-02,
         3.22795593e-01, -3.38640961e-02, -4.84301780e-02, -7.18098818e-01,
         1.5

In [16]:
svc = SVC(random_state=42)
logi = LogisticRegression(random_state=42)


def run_model(X, Y, model):
    # X는 독립변수
    # Y는 종속변수
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.2, random_state=42, stratify=Y
    )
    # 모델에 학습
    model.fit(X_train, Y_train)
    # 학습된 모델에 예측 값
    y_pred = model.predict(X_test)
    print("정확도 :", round(accuracy_score(Y_test, y_pred), 4))
    print("분류 레포트 :", classification_report(Y_test, y_pred))

In [17]:
run_model(X_embed, Y, svc)

정확도 : 0.65
분류 레포트 :               precision    recall  f1-score   support

           0       0.64      0.60      0.62        48
           1       0.65      0.69      0.67        52

    accuracy                           0.65       100
   macro avg       0.65      0.65      0.65       100
weighted avg       0.65      0.65      0.65       100



In [18]:
run_model(X_embed, Y, logi)

정확도 : 0.68
분류 레포트 :               precision    recall  f1-score   support

           0       0.67      0.67      0.67        48
           1       0.69      0.69      0.69        52

    accuracy                           0.68       100
   macro avg       0.68      0.68      0.68       100
weighted avg       0.68      0.68      0.68       100



In [19]:
run_model(X_embed2, Y, svc)

정확도 : 0.63
분류 레포트 :               precision    recall  f1-score   support

           0       0.62      0.58      0.60        48
           1       0.64      0.67      0.65        52

    accuracy                           0.63       100
   macro avg       0.63      0.63      0.63       100
weighted avg       0.63      0.63      0.63       100



- df에서 하위 10개 데이터를 이용하여 예측

In [20]:
# 학습된 모델에 예측의 값을 반환하는 함수
# 세번째 매개변수(vec_type)를 생성 -> 기본값은 'mean'
# 'tfidf' 입력이 들어온다면 벡터화 작업을 w2v + tfidf 융합한 벡터화
def predict_sentence_list(sentences, model, vec_type='mean'):
    # sentences : 문장들의 리스트
    # 문장들을 토큰화 -> 임베딩
    X_test = []
    for sent in sentences:
        # token() 함수를 호출하여 토큰화
        tokens = tokenize(sent)
        # 토큰화된 문장을 sent_embed_mean 함수에 입력하여 호출(단위 벡터의 평균)
        if vec_type == 'mean':
            vec = sent_embed_mean(tokens)
        elif vec_type == 'tfidf':
            vec = sent_embed_tfidf(tokens)
        X_test.append(vec)
    
    preds = model.predict(X_test)
    result = []
    for sent, pred in zip(sentences, preds):
        label = '긍정' if pred == 1 else '부정'
        result.append([sent, label])
    return result

In [21]:
# 모델 학습 -> 예측
X_test = df['document'].tail(10).values
run_model(X_embed, Y, svc)

predict_sentence_list(X_test, svc, vec_type='mean')

정확도 : 0.65
분류 레포트 :               precision    recall  f1-score   support

           0       0.64      0.60      0.62        48
           1       0.65      0.69      0.67        52

    accuracy                           0.65       100
   macro avg       0.65      0.65      0.65       100
weighted avg       0.65      0.65      0.65       100



[['이걸 영화라고 찎었냐?', '긍정'],
 ['http://blog.naver.com/oroblast/220215679580 나쁜 인상은 아니지만,오랫동안 기억에 남아....종종 떠올라서....조금은 사람을 피곤하게 만드는 영화. ^^',
  '긍정'],
 ['공포나 재난영화가 아니라 아예 대놓고 비급 크리쳐개그물임ㅋㅋ 음악 완전 흥겹다ㅋ 5점정도가 적당한 거 같은데 평점이 좀 높아서ㅋㅋ',
  '긍정'],
 ['For Carl.칼 세이건으로 시작해서 칼 세이건으로 끝난다.', '부정'],
 ['디케이드 다음에 더블 다음에 오즈인데 더블은 조금밖에 안나오네요.', '부정'],
 ['인간이 문제지.. 소는 뭔죄인가..', '긍정'],
 ['평점이 너무 낮아서...', '긍정'],
 ['이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?', '부정'],
 ['청춘 영화의 최고봉.방황과 우울했던 날들의 자화상', '부정'],
 ['한국 영화 최초로 수간하는 내용이 담긴 영화', '부정']]

In [22]:
run_model(X_embed2, Y, logi)
predict_sentence_list(X_test, logi, vec_type = 'tfidf')

정확도 : 0.67
분류 레포트 :               precision    recall  f1-score   support

           0       0.65      0.67      0.66        48
           1       0.69      0.67      0.68        52

    accuracy                           0.67       100
   macro avg       0.67      0.67      0.67       100
weighted avg       0.67      0.67      0.67       100



[['이걸 영화라고 찎었냐?', '긍정'],
 ['http://blog.naver.com/oroblast/220215679580 나쁜 인상은 아니지만,오랫동안 기억에 남아....종종 떠올라서....조금은 사람을 피곤하게 만드는 영화. ^^',
  '긍정'],
 ['공포나 재난영화가 아니라 아예 대놓고 비급 크리쳐개그물임ㅋㅋ 음악 완전 흥겹다ㅋ 5점정도가 적당한 거 같은데 평점이 좀 높아서ㅋㅋ',
  '긍정'],
 ['For Carl.칼 세이건으로 시작해서 칼 세이건으로 끝난다.', '부정'],
 ['디케이드 다음에 더블 다음에 오즈인데 더블은 조금밖에 안나오네요.', '부정'],
 ['인간이 문제지.. 소는 뭔죄인가..', '긍정'],
 ['평점이 너무 낮아서...', '긍정'],
 ['이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?', '부정'],
 ['청춘 영화의 최고봉.방황과 우울했던 날들의 자화상', '부정'],
 ['한국 영화 최초로 수간하는 내용이 담긴 영화', '부정']]